# 格林公式：旋度与边界

> 从「旋度」的几何直觉到格林公式 $\oint_C \mathbf{F}\cdot d\mathbf{r} = \iint_D \nabla\times\mathbf{F}\,dA$

---


## 一、旋度 (Curl)

### 直觉：小桨轮测试

把一个微型桨轮放入流场 $\mathbf{F}=(P,Q)$。如果它**转动**，说明该点有旋度。
转速越快 $\to$ 旋度越大；逆时针 $\to$ 正旋度。

### 定义（二维）

$$\nabla\times\mathbf{F} = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$$

对 $\mathbf{F}=(-y,\,x)$：$\nabla\times\mathbf{F}=1-(-1)=2$（常数）

下面图中：背景色 = 旋度值，彩色圆环 = 流线，黑色箭头 = 局部场方向。
注意箭头形成的「旋转」——这就是旋度的几何来源 🌀


In [33]:
# ========== 旋度演示：变旋度场 + 可拖桨轮 + 流动箭头 ==========
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def g(x): return 1 + 0.4*np.sin(x)
def F(x, y):
    gx = g(x); return -y*gx, x*gx
def curl_field(x, y):
    return 2*g(x) + 0.4*x*np.cos(x)

xg = np.linspace(-2.5, 2.5, 60)
yg = np.linspace(-2.5, 2.5, 60)
Xg, Yg = np.meshgrid(xg, yg)
Z_curl = curl_field(Xg, Yg)

theta = np.linspace(0, 2*np.pi, 200)
radii = np.linspace(0.4, 2.4, 8)
s_colors = ['#FF6B6B','#FFA94D','#FFD43B','#69DB7C',
            '#4DABF7','#748FFC','#DA77F2','#F783AC']

xa = np.linspace(-2, 2, 6)
ya = np.linspace(-2, 2, 6)
Xa, Ya = np.meshgrid(xa, ya)
n_arrow = len(Xa.flat)
arrow_x = Xa.flat.copy()
arrow_y = Ya.flat.copy()
wheel_r = 0.35

wx_s = widgets.FloatSlider(value=1.2, min=-2.2, max=2.2, step=0.1,
    description='桨轮 x', continuous_update=False,
    style={'description_width':'50px'}, layout=widgets.Layout(width='350px'))
wy_s = widgets.FloatSlider(value=0.8, min=-2.2, max=2.2, step=0.1,
    description='桨轮 y', continuous_update=False,
    style={'description_width':'50px'}, layout=widgets.Layout(width='350px'))
play = widgets.Play(value=0, min=0, max=36000, step=2, interval=40,
    description='', layout=widgets.Layout(width='50px'))

fig = go.FigureWidget()

# [0] 热力图
fig.add_trace(go.Heatmap(
    x=xg, y=yg, z=Z_curl, colorscale='Blues',
    showscale=True, zmin=0, zmax=3.5,
    colorbar=dict(title=dict(text='curl', side='right'))))

# [1..8] 流线
for r, clr in zip(radii, s_colors):
    fig.add_trace(go.Scatter(
        x=r*np.cos(theta), y=r*np.sin(theta),
        mode='lines', line=dict(width=2, color=clr), showlegend=False))

# [9] 全场箭头杆
U0, V0 = F(arrow_x, arrow_y)
M0 = np.sqrt(U0**2+V0**2)
Un0 = np.where(M0>0.01, U0/M0*0.18, 0)
Vn0 = np.where(M0>0.01, V0/M0*0.18, 0)
x_seg0, y_seg0 = [], []
tip_x0, tip_y0, tip_a0 = [], [], []
for i in range(n_arrow):
    if M0[i] < 0.01: continue
    x_seg0.extend([arrow_x[i], arrow_x[i]+Un0[i], None])
    y_seg0.extend([arrow_y[i], arrow_y[i]+Vn0[i], None])
    tip_x0.append(arrow_x[i]+Un0[i]); tip_y0.append(arrow_y[i]+Vn0[i])
    tip_a0.append(np.degrees(np.arctan2(Vn0[i], Un0[i])))
fig.add_trace(go.Scatter(
    x=x_seg0, y=y_seg0, mode='lines',
    line=dict(width=1.5, color='#555'), showlegend=False))
# [10] 全场箭头尖
fig.add_trace(go.Scatter(
    x=tip_x0, y=tip_y0, mode='markers',
    marker=dict(symbol='triangle-right', size=7, color='#555', angle=tip_a0),
    showlegend=False))

# [11] 桨轮附近强调箭头杆
fig.add_trace(go.Scatter(
    x=[], y=[], mode='lines',
    line=dict(width=3, color='#E74C3C'), showlegend=False))
# [12] 桨轮附近强调箭头尖
fig.add_trace(go.Scatter(
    x=[], y=[], mode='markers',
    marker=dict(symbol='triangle-right', size=9, color='#E74C3C'),
    showlegend=False))

# [13] 桨轮圆
fig.add_trace(go.Scatter(
    x=wx_s.value+wheel_r*np.cos(np.linspace(0,2*np.pi,60)),
    y=wy_s.value+wheel_r*np.sin(np.linspace(0,2*np.pi,60)),
    mode='lines', line=dict(width=2.5, color='#E74C3C', dash='dash'),
    name='桨轮', showlegend=True))
# [14] 十字H / [15] 十字V
fig.add_trace(go.Scatter(
    x=[wx_s.value-wheel_r, wx_s.value+wheel_r],
    y=[wy_s.value, wy_s.value],
    mode='lines', line=dict(width=3, color='#E74C3C'), showlegend=False))
fig.add_trace(go.Scatter(
    x=[wx_s.value, wx_s.value],
    y=[wy_s.value-wheel_r, wy_s.value+wheel_r],
    mode='lines', line=dict(width=3, color='#E74C3C'), showlegend=False))
# [16] 中心 / [17] 标注
fig.add_trace(go.Scatter(
    x=[wx_s.value], y=[wy_s.value],
    mode='markers', marker=dict(size=6, color='#E74C3C'), showlegend=False))
fig.add_trace(go.Scatter(
    x=[wx_s.value], y=[wy_s.value-0.5], mode='text',
    text=[f'curl={curl_field(wx_s.value,wy_s.value):.1f}'],
    textfont=dict(size=11, color='#E74C3C'), showlegend=False))

fig.update_layout(
    title='<b>旋度演示</b>  F=(-y·g(x), x·g(x))  旋度随 x 变化  |  ▶ 箭头沿流线前进',
    xaxis=dict(title='x', scaleanchor='y', scaleratio=1, range=[-2.8,2.8]),
    yaxis=dict(title='y', range=[-2.8,2.8]),
    width=680, height=680, margin=dict(l=50,r=50,t=50,b=50))

prev_play = 0
def refresh(*args):
    global prev_play
    wx, wy = wx_s.value, wy_s.value
    dp = play.value - prev_play; prev_play = play.value
    if dp < -10000: dp += 36000
    if dp <= 0: dp = 2

    dt = dp * 0.003  # 加快

    # 全场箭头沿流线前进
    for i in range(n_arrow):
        fx, fy = F(arrow_x[i], arrow_y[i])
        arrow_x[i] += fx * dt
        arrow_y[i] += fy * dt
        if abs(arrow_x[i]) > 2.8 or abs(arrow_y[i]) > 2.8:
            arrow_x[i] = Xa.flat[i]; arrow_y[i] = Ya.flat[i]

    U, V = F(arrow_x, arrow_y)
    M = np.sqrt(U**2+V**2)
    Un = np.where(M>0.01, U/M*0.18, 0)
    Vn = np.where(M>0.01, V/M*0.18, 0)
    x_seg, y_seg = [], []
    tip_x, tip_y, tip_a = [], [], []
    for i in range(n_arrow):
        if M[i] < 0.01: continue
        x_seg.extend([arrow_x[i], arrow_x[i]+Un[i], None])
        y_seg.extend([arrow_y[i], arrow_y[i]+Vn[i], None])
        tip_x.append(arrow_x[i]+Un[i]); tip_y.append(arrow_y[i]+Vn[i])
        tip_a.append(np.degrees(np.arctan2(Vn[i], Un[i])))
    fig.data[9].x = x_seg; fig.data[9].y = y_seg
    fig.data[10].x = tip_x; fig.data[10].y = tip_y
    fig.data[10].marker.angle = tip_a

    # 桨轮附近强调箭头
    near_r = wheel_r*1.6; n_near = 8
    near_a = np.linspace(0, 2*np.pi, n_near, endpoint=False)
    nx_seg, ny_seg = [], []; nt_x, nt_y, nt_a = [], [], []
    for a in near_a:
        nx = wx+near_r*np.cos(a); ny = wy+near_r*np.sin(a)
        fu, fv = F(nx, ny)
        mag = np.sqrt(fu**2+fv**2)
        if mag > 0.01: fu, fv = fu/mag*0.25, fv/mag*0.25
        nx_seg.extend([nx, nx+fu, None])
        ny_seg.extend([ny, ny+fv, None])
        nt_x.append(nx+fu); nt_y.append(ny+fv)
        nt_a.append(np.degrees(np.arctan2(fv, fu)))
    fig.data[11].x = nx_seg; fig.data[11].y = ny_seg
    fig.data[12].x = nt_x; fig.data[12].y = nt_y
    fig.data[12].marker.angle = nt_a

    # 桨轮
    curl_w = curl_field(wx, wy)
    wa = play.value*curl_w*0.01; ca, sa = np.cos(wa), np.sin(wa)
    fig.data[13].x = wx+wheel_r*np.cos(np.linspace(0,2*np.pi,60))
    fig.data[13].y = wy+wheel_r*np.sin(np.linspace(0,2*np.pi,60))
    fig.data[14].x = [wx-wheel_r*ca, wx+wheel_r*ca]
    fig.data[14].y = [wy-wheel_r*sa, wy+wheel_r*sa]
    fig.data[15].x = [wx+wheel_r*sa, wx-wheel_r*sa]
    fig.data[15].y = [wy-wheel_r*ca, wy+wheel_r*ca]
    fig.data[16].x = [wx]; fig.data[16].y = [wy]
    fig.data[17].x = [wx]; fig.data[17].y = [wy-0.5]
    fig.data[17].text = [f'curl={curl_w:.1f}']

wx_s.observe(refresh, 'value')
wy_s.observe(refresh, 'value')
play.observe(refresh, 'value')

display(widgets.VBox([widgets.HBox([wx_s, wy_s]), play, fig]))

### 关键观察

- 蓝色底色 = 旋度值（这里 curl=2 是常数，均匀着色）
- 彩色圆环 = 流线——如果放一个小桨轮到流场里，它会沿流线移动**同时自转**
- 红色虚线圆 + 十字 = 桨轮示意：场在上方指向右、下方指向左——
  合力产生逆时针扭矩 $\to$ 桨轮逆时针转 $\to$ 正旋度
- 如果场是 $\mathbf{F}=(y,\,x)$（curl=0），流线是双曲线，桨轮不转——试试改代码？


---

### 💡 为什么旋度 = $Q_x − P_y$？

聚焦一个微元 $[x, x+\delta x]\times[y, y+\delta y]$。
绕它走一圈（逆时针），感受向量场 $\mathbf{F}=(P,Q)$ 沿边界的推力：

$$\begin{aligned}
\text{环量} &= \underbrace{P(x,y)}_{\text{底边右推}}\cdot\delta x
              + \underbrace{Q(x+\delta x,y)}_{\text{右边上推}}\cdot\delta y \\
            &- \underbrace{P(x,y+\delta y)}_{\text{顶边左推}}\cdot\delta x
              - \underbrace{Q(x,y)}_{\text{左边下推}}\cdot\delta y \\
            &= [Q(x+\delta x)-Q(x)]\,\delta y - [P(y+\delta y)-P(y)]\,\delta x \\
            &\approx \frac{\partial Q}{\partial x}\delta x\delta y
                     - \frac{\partial P}{\partial y}\delta x\delta y
\end{aligned}$$

除以面积 $\delta x\delta y$：

$$\boxed{\nabla\times\mathbf{F} = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}}$$

**直觉**：右边比左边「上推力」大 → 逆时针转；顶边比底边「右推力」大 → 顺时针转。
两者之差就是净旋转趋势。

下面拖动滑块改变 $Q_x$ 和 $P_y$，观察四个边上的向量长度和中央旋度箭头的实时变化 🎮

In [ ]:
# ========== 微元上的旋度：Q_x - P_y（双滑块 + 变速旋转） ==========
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

dx, dy = 1.0, 0.7
x0, y0 = -dx/2, -dy/2
cx, cy = x0 + dx/2, y0 + dy/2
arrow_scale = 0.35
r_curl = 0.25

# ---- 控件 ----
dQdx_s = widgets.FloatSlider(value=2.0, min=-3, max=3, step=0.1,
    description='dQ/dx', continuous_update=True,
    style={'description_width':'50px'}, layout=widgets.Layout(width='380px'))
dPdy_s = widgets.FloatSlider(value=1.0, min=-3, max=3, step=0.1,
    description='dP/dy', continuous_update=True,
    style={'description_width':'50px'}, layout=widgets.Layout(width='380px'))
play = widgets.Play(value=0, min=0, max=36000, step=2, interval=40,
    description='', layout=widgets.Layout(width='50px'))

# ---- 构建初始 FigureWidget（8 个 trace，只创建一次） ----
init_dQ, init_dP = 2.0, 1.0
curl0 = init_dQ - init_dP
P0, Q0 = 1.5, 1.5
Pb0 = P0 - init_dP * dy/2
Pt0 = P0 + init_dP * dy/2
Qr0 = Q0 + init_dQ * dx/2
Ql0 = Q0 - init_dQ * dx/2

fig = go.FigureWidget()
# [0] 矩形
fig.add_trace(go.Scatter(
    x=[x0,x0+dx,x0+dx,x0,x0], y=[y0,y0,y0+dy,y0+dy,y0],
    mode='lines', line=dict(width=2, color='#555'), showlegend=False))
# [1] 底边 P
fig.add_trace(go.Scatter(
    x=[x0+dx/2, x0+dx/2+Pb0*arrow_scale], y=[y0,y0],
    mode='lines+markers', line=dict(width=4, color='#E74C3C'),
    marker=dict(size=8, symbol='triangle-right', color='#E74C3C'), showlegend=False))
# [2] 右边 Q
fig.add_trace(go.Scatter(
    x=[x0+dx, x0+dx], y=[y0+dy/2, y0+dy/2+Qr0*arrow_scale],
    mode='lines+markers', line=dict(width=4, color='#3498DB'),
    marker=dict(size=8, symbol='triangle-up', color='#3498DB'), showlegend=False))
# [3] 顶边 P
fig.add_trace(go.Scatter(
    x=[x0+dx/2, x0+dx/2-Pt0*arrow_scale], y=[y0+dy, y0+dy],
    mode='lines+markers', line=dict(width=4, color='#E74C3C', dash='dot'),
    marker=dict(size=8, symbol='triangle-left', color='#E74C3C'), showlegend=False))
# [4] 左边 Q
fig.add_trace(go.Scatter(
    x=[x0, x0], y=[y0+dy/2, y0+dy/2-Ql0*arrow_scale],
    mode='lines+markers', line=dict(width=4, color='#3498DB', dash='dot'),
    marker=dict(size=8, symbol='triangle-down', color='#3498DB'), showlegend=False))
# [5] 旋度弧
fig.add_trace(go.Scatter(
    x=[cx], y=[cy], mode='lines', line=dict(width=3, color='#2ECC71'),
    visible=True, showlegend=False))
# [6] 旋度箭头
fig.add_trace(go.Scatter(
    x=[cx], y=[cy], mode='lines', line=dict(width=2, color='#2ECC71'),
    fill='toself', fillcolor='rgba(46,204,113,0.4)',
    visible=True, showlegend=False))
# [7] 标注
fig.add_trace(go.Scatter(
    x=[cx], y=[cy+0.55], mode='text',
    text=['curl = 1.0'], textfont=dict(size=13, color='#333'), showlegend=False))

fig.update_layout(
    title=dict(text='<b>微元上的旋度</b>  curl = Q<sub>x</sub> - P<sub>y</sub> = 1.0',
               x=0.5, xanchor='center'),
    xaxis=dict(title='x', range=[-1.3,1.3], scaleanchor='y', scaleratio=1),
    yaxis=dict(title='y', range=[-1.1,1.3]),
    width=700, height=600, margin=dict(l=50, r=20, t=60, b=20))

# ---- 更新函数（只改 .x .y .text .visible） ----
def refresh(*args):
    dQ = dQdx_s.value; dP = dPdy_s.value
    curl = dQ - dP
    rot = np.radians(play.value * curl / 1.0)
    Pb = P0 - dP * dy/2; Pt = P0 + dP * dy/2
    Qr = Q0 + dQ * dx/2; Ql = Q0 - dQ * dx/2

    fig.data[1].x = [x0+dx/2, x0+dx/2 + Pb*arrow_scale]
    fig.data[2].y = [y0+dy/2, y0+dy/2 + Qr*arrow_scale]
    fig.data[3].x = [x0+dx/2, x0+dx/2 - Pt*arrow_scale]
    fig.data[4].y = [y0+dy/2, y0+dy/2 - Ql*arrow_scale]

    if abs(curl) > 0.01:
        span = min(abs(curl), 6) / 6 * 1.7 * np.pi
        base = np.pi/4 + rot
        arc = np.linspace(base, base+span, 80) if curl>0 else np.linspace(base, base-span, 80)
        ax_arr = cx + r_curl*np.cos(arc)
        ay_arr = cy + r_curl*np.sin(arc)
        tx = ax_arr[-1]-ax_arr[-2]; ty = ay_arr[-1]-ay_arr[-2]
        t_len = np.sqrt(tx*tx + ty*ty)
        if t_len > 0:
            tx, ty = tx/t_len, ty/t_len; aw = 0.06; px, py = -ty, tx
            ah_x = [ax_arr[-1]+aw*0.5*tx, ax_arr[-1]-aw*0.3*tx+aw*0.5*px,
                    ax_arr[-1]-aw*0.3*tx-aw*0.5*px, ax_arr[-1]+aw*0.5*tx]
            ah_y = [ay_arr[-1]+aw*0.5*ty, ay_arr[-1]-aw*0.3*ty+aw*0.5*py,
                    ay_arr[-1]-aw*0.3*ty-aw*0.5*py, ay_arr[-1]+aw*0.5*ty]
        else:
            ax_arr,ay_arr=[cx],[cy]; ah_x,ah_y=[cx],[cy]
        fig.data[5].x = list(ax_arr); fig.data[5].y = list(ay_arr)
        fig.data[5].visible = True
        fig.data[6].x = ah_x; fig.data[6].y = ah_y
        fig.data[6].visible = True
    else:
        fig.data[5].visible = False
        fig.data[6].visible = False

    fig.data[7].text = [f'curl = {dQ:.1f} - ({dP:.1f}) = {curl:.1f}']
    fig.update_layout(title=dict(
        text=f'<b>微元上的旋度</b>  curl = Q<sub>x</sub> - P<sub>y</sub> = {curl:.1f}',
        x=0.5, xanchor='center'))

dQdx_s.observe(refresh, 'value')
dPdy_s.observe(refresh, 'value')
play.observe(refresh, 'value')

display(widgets.VBox([dQdx_s, dPdy_s, play, fig]))

---

## 二、格林公式：旋度的面积分 = 边界的线积分

$$\boxed{\oint_{\partial D} \mathbf{F}\cdot d\mathbf{r} = \iint_D \nabla\times\mathbf{F}\,dA}$$

### 直观理解

把区域 $D$ 切成许多小方格：
- 每个小方格的**边界环量** $\approx$ 旋度 $\times$ 面积
- 相邻方格的共享边**互相抵消**
- 只剩下最外圈 $\to$ 总环量 = 旋度的面积分

### 3D 视角

把旋度画成 $z$ 轴高度——在区域 $D$ 上方形成一个曲面 $z=\nabla\times\mathbf{F}$。
格林公式说：**这个曲面下方的体积 = 边界上的环量**。

下面以 $\mathbf{F}=(0,\,x^2/2)$（curl $=x$）在矩形 $[0,2]\times[-1,1]$ 上演示：


In [ ]:
# ========== 格林公式 3D：旋度曲面 + 边界 ==========
import numpy as np
import plotly.graph_objects as go

# 区域 D = [0, 2] x [-1, 1]，F = (0, x^2/2)，curl = x
x_r = np.linspace(0, 2, 40)
y_r = np.linspace(-1, 1, 30)
Xr, Yr = np.meshgrid(x_r, y_r)
Zr = Xr  # curl = x

# ---- 边界（矩形，逆时针） ----
bx_b = np.linspace(0, 2, 60);    by_b = np.full_like(bx_b, -1)
bx_r = np.full_like(by_r := np.linspace(-1, 1, 60), 2);  by_r = by_r
bx_t = np.linspace(2, 0, 60);    by_t = np.full_like(bx_t, 1)
bx_l = np.full_like(by_l := np.linspace(1, -1, 60), 0);  by_l = by_l
bx_all = np.concatenate([bx_b, bx_r, bx_t, bx_l])
by_all = np.concatenate([by_b, by_r, by_t, by_l])
bz_surf = bx_all  # curl at boundary = x

# ---- 3D 图 ----
fig_green = go.Figure()

# 1) 旋度曲面 z = x
fig_green.add_trace(go.Surface(
    x=Xr, y=Yr, z=Zr, colorscale='Blues', opacity=0.6,
    cmin=0, cmax=2, showscale=True,
    colorbar=dict(title=dict(text='curl = x', side='right')),
    name='curl 曲面',
))

# 2) 底部边界 (z=0)
fig_green.add_trace(go.Scatter3d(
    x=bx_all, y=by_all, z=np.zeros_like(bx_all),
    mode='lines', line=dict(width=4, color='#E74C3C'),
    name='partial D (z=0)',
))

# 3) 边界抬升到曲面
fig_green.add_trace(go.Scatter3d(
    x=bx_all, y=by_all, z=bz_surf,
    mode='lines', line=dict(width=3, color='#E74C3C', dash='dot'),
    name='partial D 在曲面上的投影',
))

# 4) 四边「窗帘」——体积可视化
for bx, by in [(bx_b, by_b), (bx_r, by_r), (bx_t, by_t), (bx_l, by_l)]:
    for i in range(len(bx)-1):
        fig_green.add_trace(go.Mesh3d(
            x=[bx[i], bx[i+1], bx[i+1], bx[i]],
            y=[by[i], by[i+1], by[i+1], by[i]],
            z=[0, 0, bx[i+1], bx[i]],
            color='lightblue', opacity=0.12, showlegend=False,
        ))

# 5) 边界方向箭头
arrow_pts = [(1,-1,0.1,0), (2,0,0,0.1), (1,1,-0.1,0), (0,0,0,-0.1)]
for ax, ay, adx, ady in arrow_pts:
    fig_green.add_trace(go.Cone(
        x=[ax], y=[ay], z=[0.15], u=[adx], v=[ady], w=[0],
        sizemode='absolute', sizeref=0.12,
        colorscale=[[0,'#E74C3C'],[1,'#E74C3C']], showscale=False,
        showlegend=False,
    ))

# 6) 标注
fig_green.add_trace(go.Scatter3d(
    x=[1], y=[0], z=[1.2], mode='text',
    text=['体积 = 环量 = 4'],
    textfont=dict(size=16, color='#333'), showlegend=False,
))

fig_green.update_layout(
    title=dict(
        text='<b>格林公式 3D</b>  F=(0, x^2/2)  curl=x  D=[0,2]x[-1,1]<br>'
             '<sup>曲面下体积 = area integral of curl = 4  =  边界环量 = line integral of F</sup>',
        x=0.5, xanchor='center',
    ),
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='curl',
        camera=dict(eye=dict(x=2.0, y=-2.5, z=1.5)),
        aspectratio=dict(x=1.2, y=1, z=0.8),
    ),
    width=850, height=600,
    margin=dict(l=0, r=0, t=70, b=0),
)

fig_green.show()


In [ ]:
# ========== 数值验证 ==========
import numpy as np

# 区域 D = [0,2] x [-1,1]，F = (0, x^2/2)

# ---- 面积分 ----
n_grid = 200
xv = np.linspace(0, 2, n_grid)
yv = np.linspace(-1, 1, n_grid)
dx, dy = xv[1]-xv[0], yv[1]-yv[0]
Xv, Yv = np.meshgrid(xv, yv)
area_int = np.sum(Xv) * dx * dy   # curl = x
print(f'Area integral = {area_int:.6f}  (analytic = 4)')

# ---- 线积分 ----
n_b = 500
line_int = 0.0

# 右边 x=2, F=(0, 2), dr=(0, dy)
ys = np.linspace(-1, 1, n_b)
line_int += np.sum(2 * (ys[1]-ys[0]))
# 其他三边积分为 0（P=0 且水平边 dy=0；x=0 时 Q=0）

print(f'Line integral  = {line_int:.6f}  (analytic = 4)')
print(f'Relative error = {abs(area_int-line_int)/abs(line_int)*100:.4f}%')


---

## 三、总结

| 概念 | 含义 | 可视化 |
|------|------|--------|
| **旋度** $\nabla\times\mathbf{F}$ | 局部「旋转倾向」= 桨轮转速 | 2D 热力图 + 流线 + 桨轮 |
| **线积分** $\oint_C\mathbf{F}\cdot d\mathbf{r}$ | 沿闭合路径做的「功」 | 边界上的红色路径 |
| **面积分** $\iint_D\nabla\times\mathbf{F}\,dA$ | 旋度在区域上的累积 | 3D 曲面下的体积 |
| **格林公式** | 两者相等——内部旋度之和 = 边界环量 | 体积 = 环量 |

> 一句话：区域内部所有小桨轮的旋转加起来，等于绕边界走一圈感受到的「风」。
> 相邻格子共享边上的风互相抵消，只剩最外圈 $\to$ 格林公式。
